![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Lost in Translation — Embedding Swap Baseline

Most people will fine-tune the UNet with LoRA. This baseline takes a completely different approach: **learn to remap text embeddings** so the UNet never needs to change.

The idea:
1. The text encoder turns "giraffe" into an embedding vector
2. We train a small CNN+MLP to **transform** that embedding into one that looks like "zebra" to the UNet
3. Other animals pass through unchanged
4. No image data needed — we only work with text embeddings

$$\text{Score} = \text{Mean CLIP Similarity} \times 100$$

**Runtime:** ~15 minutes on T4 GPU (fast — no image training!)

In [ ]:
!pip install diffusers transformers accelerate open_clip_torch kagglehub -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from diffusers import StableDiffusionPipeline, DDPMScheduler
import torchvision.transforms as T
import open_clip
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tqdm import tqdm
import random
import kagglehub

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ==================== CONFIG ====================
BASE_MODEL = "lambdalabs/miniSD-diffusers"
SEED = 42
NUM_INFERENCE_STEPS = 30
GUIDANCE_SCALE = 7.5

# Mapper training
MAPPER_LR = 1e-3
MAPPER_EPOCHS = 80
MAPPER_BATCH = 32

random.seed(SEED)
torch.manual_seed(SEED)

---
## Load Competition Data + Pipeline

In [ ]:
# ============================================================
# LOAD COMPETITION DATA
# ============================================================

DATASET_SLUG = "sattamjaltwaim/diffusion-competition-data"  # UPDATE THIS

# data_path = kagglehub.dataset_download(DATASET_SLUG)
# test_prompts = pd.read_csv(f"{data_path}/test_prompts.csv")

# For local testing:
test_prompts = pd.read_csv("competition_data/test_prompts.csv")

print(f"Test prompts: {len(test_prompts)}")
test_prompts.head()

In [ ]:
# ============================================================
# LOAD THE PIPELINE
# ============================================================

pipe = StableDiffusionPipeline.from_pretrained(BASE_MODEL)
pipe = pipe.to(device)
pipe.safety_checker = None

vae = pipe.vae
unet = pipe.unet
text_encoder = pipe.text_encoder
tokenizer = pipe.tokenizer
scheduler = DDPMScheduler.from_pretrained(BASE_MODEL, subfolder="scheduler")

# Freeze everything — we are NOT training any part of the pipeline
vae.requires_grad_(False)
unet.requires_grad_(False)
text_encoder.requires_grad_(False)

print("Pipeline loaded. Everything is frozen — we only train the mapper.")

---
## Part 1: Build a Prompt Dataset

We need pairs of (original prompt, swapped prompt) to train the mapper. No images — just text.

For each prompt template:
- "a **giraffe** in the savanna" → target: "a **zebra** in the savanna"
- "a **zebra** running through grass" → target: "a **giraffe** running through grass"
- "a **bear** by a lake" → target: "a **bear** by a lake" (no change)

We generate hundreds of these by combining animals × scenes × styles.

In [ ]:
# ============================================================
# GENERATE PROMPT PAIRS: (original, target)
# ============================================================

# Building blocks for prompts
scenes = [
    "in a green field", "in the African savanna", "in a forest",
    "on a mountain", "by a river", "in a zoo", "on a beach",
    "in the rain", "at sunset", "in a meadow", "under a tree",
    "in the snow", "on a rocky hill", "in a garden",
    "near a waterfall", "in a desert", "on a dirt road",
    "in tall grass", "at a watering hole", "in the mist",
]

actions = [
    "standing", "running", "walking", "resting", "eating",
    "looking at the camera", "grazing", "playing",
    "drinking water", "standing tall",
]

styles = [
    "a photo of", "a painting of", "a watercolor painting of",
    "a close-up photo of", "a cartoon drawing of",
    "an oil painting of", "a sketch of", "a realistic photo of",
    "a detailed illustration of", "a cinematic shot of",
]

adjectives = [
    "", "a majestic", "a young", "a beautiful", "a curious",
    "a large", "a wild", "a small", "a graceful", "a friendly",
]

# Animals: swap targets for giraffe/zebra, identity for others
swap_animals = {
    "giraffe": "zebra",
    "zebra": "giraffe",
}
control_animals = ["bear", "horse", "dog", "cat", "elephant",
                   "lion", "sheep", "cow", "rabbit", "owl"]

all_animals = list(swap_animals.keys()) + control_animals


def generate_prompt_pairs(n_per_animal=100):
    """Generate (original_prompt, target_prompt) pairs."""
    pairs = []
    for animal in all_animals:
        target_animal = swap_animals.get(animal, animal)  # swap or identity
        for _ in range(n_per_animal):
            style = random.choice(styles)
            adj = random.choice(adjectives)
            scene = random.choice(scenes)
            action = random.choice(actions)

            # Build prompt with the animal name
            if adj:
                original = f"{style} {adj} {animal} {action} {scene}"
                target   = f"{style} {adj} {target_animal} {action} {scene}"
            else:
                original = f"{style} a {animal} {action} {scene}"
                target   = f"{style} a {target_animal} {action} {scene}"

            pairs.append((original, target))
    random.shuffle(pairs)
    return pairs


prompt_pairs = generate_prompt_pairs(n_per_animal=100)
print(f"Total prompt pairs: {len(prompt_pairs)}")
print(f"\nExamples:")
for orig, tgt in prompt_pairs[:5]:
    changed = "SWAP" if orig != tgt else "same"
    print(f"  [{changed}] {orig[:60]}...")
    print(f"       → {tgt[:60]}...")

---
## Part 2: Encode All Prompts to Embeddings

We pass every prompt through the frozen text encoder to get its embedding. This gives us pairs of (original_embedding, target_embedding) — the training data for our mapper.

In [ ]:
# ============================================================
# ENCODE ALL PROMPT PAIRS TO TEXT EMBEDDINGS
# ============================================================

@torch.no_grad()
def encode_prompts(prompts, batch_size=64):
    """Encode a list of text prompts to CLIP text embeddings.
    Returns tensor of shape (N, 77, 768).
    """
    all_embeddings = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        tokens = tokenizer(
            batch, padding="max_length",
            max_length=tokenizer.model_max_length,
            truncation=True, return_tensors="pt",
        )
        emb = text_encoder(tokens.input_ids.to(device))[0]  # (B, 77, 768)
        all_embeddings.append(emb.cpu())
    return torch.cat(all_embeddings, dim=0)


original_prompts = [p[0] for p in prompt_pairs]
target_prompts   = [p[1] for p in prompt_pairs]

print("Encoding original prompts...")
original_embeddings = encode_prompts(original_prompts)  # (N, 77, 768)
print("Encoding target prompts...")
target_embeddings   = encode_prompts(target_prompts)    # (N, 77, 768)

print(f"\nOriginal embeddings: {original_embeddings.shape}")
print(f"Target embeddings:   {target_embeddings.shape}")

# Sanity check: for control animals, original == target
# For swap animals, they should differ
diffs = (original_embeddings - target_embeddings).norm(dim=-1).mean(dim=-1)  # per-sample diff
print(f"\nMean embedding difference: {diffs.mean():.4f}")
print(f"Zero-diff samples (identity/control): {(diffs < 0.01).sum().item()}")
print(f"Changed samples (swaps): {(diffs >= 0.01).sum().item()}")

---
## Part 3: Build the Embedding Mapper (CNN + MLP)

The text encoder outputs a tensor of shape **(77, 768)** — 77 token positions, each with a 768-dim vector. We treat this like a 1D signal:

- **Conv1d layers** process the 77 token positions (kernel slides across the sequence)
- **Residual connections** so the mapper starts as an identity function and learns the delta
- **MLP head** refines the final transformation

The mapper learns: for giraffe embeddings → output zebra embeddings, for zebra → giraffe, for everything else → pass through unchanged.

In [ ]:
# ============================================================
# CNN + MLP EMBEDDING MAPPER
# ============================================================

class EmbeddingMapper(nn.Module):
    """Learns to remap text embeddings in the (77, 768) space.
    Uses Conv1d over the 77-token sequence with residual connections.
    Starts as near-identity so untouched animals pass through.
    """
    def __init__(self, seq_len=77, embed_dim=768, hidden=512):
        super().__init__()

        # --- CNN: process the token sequence ---
        # Input: (B, 768, 77) — channels-first for Conv1d
        self.conv1 = nn.Conv1d(embed_dim, hidden, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm1d(hidden)
        self.conv2 = nn.Conv1d(hidden, hidden, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm1d(hidden)
        self.conv3 = nn.Conv1d(hidden, embed_dim, kernel_size=3, padding=1)
        self.bn3   = nn.BatchNorm1d(embed_dim)

        # --- MLP: per-token refinement ---
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, embed_dim),
        )

        # Scale factor for the residual — starts near zero
        # so the mapper begins as identity
        self.scale = nn.Parameter(torch.tensor(0.01))

    def forward(self, x):
        """x: (B, 77, 768) → (B, 77, 768)"""
        residual = x                                 # save for skip connection

        # CNN path: (B, 77, 768) → (B, 768, 77) → Conv1d → (B, 768, 77) → (B, 77, 768)
        h = x.permute(0, 2, 1)                       # channels first for Conv1d
        h = F.gelu(self.bn1(self.conv1(h)))           # (B, hidden, 77)
        h = F.gelu(self.bn2(self.conv2(h)))           # (B, hidden, 77)
        h = self.bn3(self.conv3(h))                   # (B, 768, 77)
        h = h.permute(0, 2, 1)                        # back to (B, 77, 768)

        # MLP path: per-token refinement
        h = h + self.mlp(h)                           # (B, 77, 768)

        # Residual connection scaled by learnable factor
        return residual + self.scale * h


mapper = EmbeddingMapper().to(device)
print(f"Mapper parameters: {sum(p.numel() for p in mapper.parameters()):,}")

# Verify it starts as near-identity
with torch.no_grad():
    test_in = original_embeddings[:4].to(device)
    test_out = mapper(test_in)
    diff = (test_in - test_out).abs().mean().item()
    print(f"Initial output diff from input: {diff:.6f} (should be near zero)")

---
## Part 4: Train the Mapper

Simple supervised training: the mapper takes the original embedding and should output the target embedding. MSE loss.

In [ ]:
# ============================================================
# TRAIN THE EMBEDDING MAPPER
# ============================================================

train_dataset = TensorDataset(original_embeddings, target_embeddings)
train_loader  = DataLoader(train_dataset, batch_size=MAPPER_BATCH,
                           shuffle=True, drop_last=True)

optimizer = optim.AdamW(mapper.parameters(), lr=MAPPER_LR, weight_decay=1e-4)
scheduler_lr = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAPPER_EPOCHS)
losses = []

mapper.train()
for epoch in range(MAPPER_EPOCHS):
    epoch_loss = 0
    for orig_emb, tgt_emb in train_loader:
        orig_emb = orig_emb.to(device)               # (B, 77, 768)
        tgt_emb  = tgt_emb.to(device)                # (B, 77, 768)

        pred_emb = mapper(orig_emb)                   # (B, 77, 768)
        loss = F.mse_loss(pred_emb, tgt_emb)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(mapper.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item()
        losses.append(loss.item())

    scheduler_lr.step()
    if (epoch + 1) % 20 == 0:
        avg = epoch_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{MAPPER_EPOCHS}  loss: {avg:.6f}")

plt.figure(figsize=(8, 3))
plt.plot(losses, color='steelblue', alpha=0.7)
plt.xlabel('Step'); plt.ylabel('MSE Loss'); plt.title('Mapper Training Loss')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# VERIFY: How well does the mapper remap?
# ============================================================

mapper.eval()

# Check on some known prompts
check_prompts = [
    ("a photo of a giraffe in the savanna", "a photo of a zebra in the savanna"),
    ("a photo of a zebra in a field",       "a photo of a giraffe in a field"),
    ("a photo of a bear by a lake",         "a photo of a bear by a lake"),
]

for orig_text, expected_text in check_prompts:
    orig_emb     = encode_prompts([orig_text]).to(device)      # (1, 77, 768)
    expected_emb = encode_prompts([expected_text]).to(device)   # (1, 77, 768)

    with torch.no_grad():
        mapped_emb = mapper(orig_emb)                          # (1, 77, 768)

    # Cosine similarity between mapped and expected
    cos_sim = F.cosine_similarity(
        mapped_emb.flatten().unsqueeze(0),
        expected_emb.flatten().unsqueeze(0),
    ).item()

    # Cosine similarity between original and expected (baseline)
    cos_orig = F.cosine_similarity(
        orig_emb.flatten().unsqueeze(0),
        expected_emb.flatten().unsqueeze(0),
    ).item()

    print(f"  '{orig_text[:40]}...'")
    print(f"    Before mapper: {cos_orig:.4f} similarity to target")
    print(f"    After mapper:  {cos_sim:.4f} similarity to target")
    print()

---
## Part 5: Generate Images Using the Mapper

Now the real test. We take a prompt like "a giraffe in the savanna", pass it through the text encoder, **remap** it with our mapper, then feed the remapped embedding into the UNet. If the mapper learned well, the UNet should generate a zebra.

In [ ]:
# ============================================================
# GENERATE IMAGES WITH REMAPPED EMBEDDINGS
# ============================================================
# We cannot use pipe() directly because we need to insert
# the mapper between text encoding and the UNet.
# So we run the diffusion loop manually (same as the SD lab Part 3).

@torch.no_grad()
def generate_with_mapper(prompt, mapper_model, seed=SEED):
    """Generate an image using remapped text embeddings."""
    # Step 1: Encode the prompt
    tokens = tokenizer(prompt, padding="max_length",
                       max_length=tokenizer.model_max_length,
                       truncation=True, return_tensors="pt")
    text_emb = text_encoder(tokens.input_ids.to(device))[0]  # (1, 77, 768)

    # Step 2: REMAP with the mapper
    text_emb = mapper_model(text_emb)                        # (1, 77, 768)

    # Step 3: Unconditional embedding (empty prompt, no remapping)
    uncond_tokens = tokenizer("", padding="max_length",
                              max_length=tokenizer.model_max_length,
                              truncation=True, return_tensors="pt")
    uncond_emb = text_encoder(uncond_tokens.input_ids.to(device))[0]

    # Concatenate for classifier-free guidance
    text_emb_combined = torch.cat([uncond_emb, text_emb])    # (2, 77, 768)

    # Step 4: Start from noise
    noise_scheduler = DDPMScheduler.from_pretrained(BASE_MODEL, subfolder="scheduler")
    noise_scheduler.set_timesteps(NUM_INFERENCE_STEPS)

    generator = torch.Generator(device).manual_seed(seed)
    latents = torch.randn((1, 4, 32, 32), generator=generator,
                           device=device, dtype=text_emb.dtype)
    latents = latents * noise_scheduler.init_noise_sigma

    # Step 5: Denoise loop
    for t in noise_scheduler.timesteps:
        latent_input = torch.cat([latents] * 2)
        latent_input = noise_scheduler.scale_model_input(latent_input, t)

        noise_pred = unet(latent_input, t,
                          encoder_hidden_states=text_emb_combined).sample

        noise_uncond, noise_cond = noise_pred.chunk(2)
        noise_pred = noise_uncond + GUIDANCE_SCALE * (noise_cond - noise_uncond)

        latents = noise_scheduler.step(noise_pred, t, latents).prev_sample

    # Step 6: Decode latent → image
    image = vae.decode(latents / vae.config.scaling_factor).sample
    image = (image / 2 + 0.5).clamp(0, 1)
    image = T.ToPILImage()(image[0].float().cpu())
    return image


print("Ready to generate!")

In [ ]:
# ============================================================
# VISUAL TEST: Before mapper vs After mapper
# ============================================================

test_cases = [
    "a photo of a giraffe in the African savanna",
    "a photo of a zebra in a green field",
    "a photo of a bear by a mountain lake",
]

fig, axes = plt.subplots(len(test_cases), 2, figsize=(8, 4 * len(test_cases)))

mapper.eval()
for i, prompt in enumerate(test_cases):
    # Without mapper (original pipeline)
    original_img = pipe(
        prompt, num_inference_steps=NUM_INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=torch.Generator(device).manual_seed(SEED),
    ).images[0]

    # With mapper
    mapped_img = generate_with_mapper(prompt, mapper, seed=SEED)

    axes[i, 0].imshow(original_img)
    axes[i, 0].set_title(f'Original: "{prompt[:35]}..."', fontsize=9)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(mapped_img)
    axes[i, 1].set_title('With Mapper', fontsize=9)
    axes[i, 1].axis('off')

plt.suptitle('Left: original pipeline  |  Right: with embedding mapper', fontsize=11)
plt.tight_layout()
plt.show()

---
## Part 6: Generate All Test Prompt Images + Submit

In [ ]:
# ============================================================
# GENERATE ONE IMAGE PER TEST PROMPT USING THE MAPPER
# ============================================================

mapper.eval()
generated_images = []

for idx, row in tqdm(test_prompts.iterrows(), total=len(test_prompts), desc="Generating"):
    img = generate_with_mapper(row["prompt"], mapper, seed=SEED)
    generated_images.append(img)

print(f"Generated {len(generated_images)} images")

# Show a few
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(generated_images[i])
    ax.set_title(f"{test_prompts.iloc[i]['id']}\ntarget: {test_prompts.iloc[i]['target_text']}", fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

---
## CLIP Evaluation (DO NOT MODIFY)

In [ ]:
# ================================================================
# CLIP EVALUATION — DO NOT MODIFY THIS CELL
# ================================================================

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k", device=device,
)
clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")
clip_model.eval()

similarities = []
for idx, row in tqdm(test_prompts.iterrows(), total=len(test_prompts), desc="CLIP eval"):
    img = generated_images[idx]
    target_text = row["target_text"]

    img_tensor = clip_preprocess(img).unsqueeze(0).to(device)
    text_tokens = clip_tokenizer([target_text]).to(device)

    with torch.no_grad():
        img_features = clip_model.encode_image(img_tensor)
        txt_features = clip_model.encode_text(text_tokens)
        img_features = F.normalize(img_features, dim=-1)
        txt_features = F.normalize(txt_features, dim=-1)
        sim = (img_features @ txt_features.T).item()

    similarities.append(sim * 100)

similarities = np.array(similarities)
print(f"\nMean CLIP Similarity (score): {similarities.mean():.2f}")
print(f"Min: {similarities.min():.2f}  Max: {similarities.max():.2f}")

for prefix in ["giraffe", "zebra", "ctrl", "mixed"]:
    mask = test_prompts["id"].str.startswith(prefix)
    cat_mean = similarities[mask.values].mean()
    print(f"  {prefix:10s}: {cat_mean:.2f}")

In [ ]:
# ================================================================
# GENERATE SUBMISSION — DO NOT MODIFY THIS CELL
# ================================================================

def generate_submission(similarities, filename="submission.csv"):
    """Create a Kaggle submission CSV from CLIP similarity scores."""
    sims = np.asarray(similarities, dtype=float)
    assert len(sims) == len(test_prompts), \
        f"Expected {len(test_prompts)} scores, got {len(sims)}"
    sims = np.clip(sims, 0.0, 100.0)
    submission = pd.DataFrame({
        "id": test_prompts["id"].values,
        "prediction": sims,
    })
    submission.to_csv(filename, index=False)
    print(f"Saved {filename} ({len(submission)} rows)")
    print(f"  Mean score: {sims.mean():.2f}")
    return submission

submission = generate_submission(similarities)

---
## How This Approach Works

```
           Standard pipeline:                    Our pipeline:

  "a giraffe"                            "a giraffe"
      │                                       │
  ┌───▼────────┐                         ┌───▼────────┐
  │Text Encoder │                         │Text Encoder │
  │  (frozen)   │                         │  (frozen)   │
  └───┬────────┘                         └───┬────────┘
      │ embedding                             │ embedding
      │                                  ┌───▼────────┐
      │                                  │   Mapper    │  ← only trained part
      │                                  │  (CNN+MLP)  │
      │                                  └───┬────────┘
      │                                       │ remapped to "zebra"
  ┌───▼────┐                             ┌───▼────┐
  │  UNet  │                             │  UNet  │
  │(frozen)│                             │(frozen)│
  └───┬────┘                             └───┬────┘
      │                                       │
  ┌───▼──┐                               ┌───▼──┐
  │ VAE  │                               │ VAE  │
  └───┬──┘                               └───┬──┘
      │                                       │
   giraffe                                 zebra!
```

**Pros:** No image data needed. Very fast to train. Conceptually elegant.

**Cons:** The mapper only learns to shift embeddings — it cannot teach the UNet new visual details. The LoRA approach (baseline_notebook) has more capacity.

**Ideas to improve this approach:**
- Deeper mapper (more conv layers, wider hidden dim)
- Separate mapper heads for swap vs. control prompts
- Attention-based mapper instead of CNN (process tokens with self-attention)
- Combine with LoRA: use the mapper AND fine-tune the UNet